In [20]:
import pandas as pd
import networkx as nx
import ast

def clean_and_build_graphs(file_path):
    # 1. Load the dataset
    print("Loading data...")
    df = pd.read_csv(file_path)

    # 2. Convert 'Connections' from string "[1, 2]" to Python list [1, 2]
    # Using ast.literal_eval is safer than eval()
    print("Parsing connections...")
    df['Connections'] = df['Connections'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else [])

    # 3. Group by Population
    # There are 130 simulations; we treat each as a separate graph
    populations = df['Population'].unique()
    graph_dict = {}

    print(f"Building {len(populations)} individual networks...")
    
    for pop_id in populations:
        # Filter data for this specific simulation
        pop_data = df[df['Population'] == pop_id]
        
        # Initialize an Undirected Graph
        G = nx.Graph()
        
        # Add nodes with attributes (Age, Constitution, Behaviour)
        for _, row in pop_data.iterrows():
            G.add_node(row['ID'], 
                       age=row['Age'], 
                       const=row['Constitution'], 
                       behav=row['Behaviour'],
                       is_index=row['Index_Patient'],
                       target=row.get('Infected', None))
            
            # Add edges based on the connections list
            for connection in row['Connections']:
                # The simulation is undirected, so we just add the edge
                G.add_edge(row['ID'], connection)
        
        graph_dict[pop_id] = G

    print("Graphs built successfully.")
    return df, graph_dict

# --- EXECUTION ---
# df, graphs = clean_and_build_graphs('train.csv')

# Example: Accessing the graph for the first population
# first_pop_id = df['Population'].unique()[0]
# first_graph = graphs[first_pop_id]
# print(f"Nodes: {first_graph.number_of_nodes()}, Edges: {first_graph.number_of_edges()}")

In [21]:
df, graphs = clean_and_build_graphs(r'Datasets\train.csv')

Loading data...
Parsing connections...
Building 130 individual networks...
Graphs built successfully.


In [22]:
first_pop_id = df['Population'].unique()[0]
first_graph = graphs[first_pop_id]
print(f"Nodes: {first_graph.number_of_nodes()}, Edges: {first_graph.number_of_edges()}")

Nodes: 5000, Edges: 4999


In [23]:
def export_edge_list(graph_dict):
    all_edges = []
    for pop_id, G in graph_dict.items():
        for u, v in G.edges():
            all_edges.append({'Population': pop_id, 'Source': u, 'Target': v})
    
    edge_df = pd.DataFrame(all_edges)
    edge_df.to_csv(r'Datasets\network_edges.csv', index=False)
    print("Edge list exported for the Network Analysis.")

# export_edge_list(graphs)

In [24]:
export_edge_list(graphs)

Edge list exported for the Network Analysis.


In [25]:
# Network Analysis

def extract_network_features(df, graph_dict):
    print("Starting Feature Extraction...")
    all_pop_features = []

    for pop_id, G in graph_dict.items():
        # 1. Identify the Index Patient for this population
        # We look for the node where the 'is_index' attribute we set earlier is 1
        index_node = [n for n, feat in G.nodes(data=True) if feat.get('is_index') == 1]
        
        # 2. Calculate Shortest Path from Index Patient to everyone
        # if the graph is disconnected, some distances might be infinite
        distances = {}
        if index_node:
            idx = index_node[0]
            # computes distances to all reachable nodes
            distances = nx.single_source_shortest_path_length(G, idx)
        
        # 3. Calculate Centrality Measures
        degree_cent = nx.degree_centrality(G)  # Number of connections
        clustering = nx.clustering(G)          # Local group density
        
        # 4. Map features back to the rows for this population
        pop_df = df[df['Population'] == pop_id].copy()
        
        pop_df['dist_to_index'] = pop_df['ID'].map(distances).fillna(-1) # -1 if unreachable
        pop_df['degree_centrality'] = pop_df['ID'].map(degree_cent)
        pop_df['clustering_coeff'] = pop_df['ID'].map(clustering)
        
        all_pop_features.append(pop_df)

    # Combine everything back into one master dataframe
    enriched_df = pd.concat(all_pop_features)
    print("Feature Extraction Complete!")
    return enriched_df

# --- EXECUTION ---
# enriched_df = extract_network_features(df, graphs)
# enriched_df.to_csv('enriched_train_data.csv', index=False)

In [26]:
enriched_df = extract_network_features(df, graphs)
print('Extraction Complete')
enriched_df.to_csv('Datasets\enriched_train_data.csv', index=False)
print('Saved')

Starting Feature Extraction...
Feature Extraction Complete!
Extraction Complete
Saved


In [27]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

def run_full_ml_pipeline(enriched_train_path, enriched_test_path):
    # 1. Load the data created by Participant 2
    train_df = pd.read_csv(enriched_train_path)
    test_df = pd.read_csv(enriched_test_path)

    # 2. Define Features
    # Dropping non-predictive or target columns
    drop_cols = ['Infected', 'ID', 'Population', 'Connections']
    X = train_df.drop(columns=[c for c in drop_cols if c in train_df.columns])
    y = train_df['Infected']
    groups = train_df['Population']

    # 3. Categorical Handling
    # Explicitly identifying categorical columns for LightGBM
    if 'Behaviour' in X.columns:
        X['Behaviour'] = X['Behaviour'].astype('category')
    
    # 4. Initialize GroupKFold (5 Folds is standard)
    gkf = GroupKFold(n_splits=5)
    
    models = []
    oof_preds = np.zeros(len(X)) # Out-of-fold predictions to check internal score

    print(f"Training on features: {list(X.columns)}")

    for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups=groups)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        # 5. Define the Model
        # We use a lower learning rate and more estimators for better AUC
        model = lgb.LGBMClassifier(
            n_estimators=2000,
            learning_rate=0.03,
            num_leaves=31,
            max_depth=-1,
            min_child_samples=20,
            importance_type='gain', # This tells us which features provide the most "info"
            metric='auc',
            verbosity=-1,
            random_state=42
        )

        # 6. Fit with Early Stopping
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(stopping_rounds=100)]
        )

        # Record results
        fold_preds = model.predict_proba(X_val)[:, 1]
        oof_preds[val_idx] = fold_preds
        models.append(model)
        
        fold_auc = roc_auc_score(y_val, fold_preds)
        print(f"--- Fold {fold+1} | AUC: {fold_auc:.4f} ---")

    # Final Internal Score
    overall_auc = roc_auc_score(y, oof_preds)
    print(f"\nOverall Cross-Validation AUC: {overall_auc:.4f}")

    # 7. Create Submission
    print("\nGenerating Test Predictions...")
    X_test = test_df.drop(columns=[c for c in drop_cols if c in test_df.columns])
    
    # Ensure test categories match train
    if 'Behaviour' in X_test.columns:
        X_test['Behaviour'] = X_test['Behaviour'].astype('category')

    # Average predictions from all 5 models for a stable result
    test_probs = np.zeros(len(X_test))
    for model in models:
        test_probs += model.predict_proba(X_test)[:, 1]
    test_probs /= len(models)

    submission = pd.DataFrame({
        'id': test_df['ID'],
        'Infected': test_probs
    })
    
    submission.to_csv('final_submission.csv', index=False)
    print("Success! 'final_submission.csv' is ready.")
    
    return models, X.columns

# To run it:
# models, features = run_full_ml_pipeline('enriched_train.csv', 'enriched_test.csv')

In [28]:
models, features = run_full_ml_pipeline('Datasets\enriched_train_data.csv', 'Datasets\enriched_test_data.csv')

Training on features: ['id', 'Index_Patient', 'Age', 'Constitution', 'Behaviour', 'dist_to_index', 'degree_centrality', 'clustering_coeff']
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[2]	valid_0's auc: 0.485118
--- Fold 1 | AUC: 0.4851 ---
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[127]	valid_0's auc: 0.541114
--- Fold 2 | AUC: 0.5411 ---
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[865]	valid_0's auc: 0.652977
--- Fold 3 | AUC: 0.6530 ---
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[8]	valid_0's auc: 0.633417
--- Fold 4 | AUC: 0.6334 ---
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[4]	valid_0's auc: 0.639384
--- Fold 5 | AUC: 0.6394 ---

Overall Cross-Validation AUC: 0.5853

Generating Test Predictions...
Success! 'final_